**1 — Set up Required Dependencies**


In [1]:
!pip install -U --quiet \
    torch==2.5.1 \
    datasets==2.17.0 \
    transformers==4.38.2 \
    evaluate==0.4.0 \
    peft==0.3.0 \
    trl==0.4.2 \
    rouge_score==0.1.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━

In [2]:
!pip install -U --quiet \
    numpy==1.26.4 \
    pandas==2.2.2 \
    pyarrow==18.1.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 64.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
torchvision 0.26.0+cpu requires torch==2.11.

In [1]:
import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import evaluate
import peft
import trl

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)

numpy: 1.26.4
pandas: 2.2.2
torch: 2.5.1+cu124
transformers: 4.38.2
datasets: 2.17.0
evaluate: 0.4.0
peft: 0.3.0
trl: 0.4.2


**Notes**

* Colab showed dependency warnings because it already has many pre-installed libraries with different version requirements. These warnings do not necessarily mean the lab failed. After restarting the runtime, the imports worked correctly, so the environment is ready for the lab.

* The required libraries were installed and verified successfully. The environment is ready to work with FLAN-T5, PEFT, TRL, PPO, and toxicity evaluation.

**2 - Load FLAN-T5 Model, Prepare Reward Model and Toxicity Evaluato**

**2.1 — Load Data and FLAN-T5 Model**

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore")

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    GenerationConfig
)

from datasets import load_dataset
from peft import PeftModel, PeftConfig, LoraConfig, TaskType

from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead
from trl import create_reference_model
from trl.core import LengthSampler

import torch
import evaluate
import numpy as np
import pandas as pd

from tqdm import tqdm
tqdm.pandas()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
model_name = "google/flan-t5-base"
huggingface_dataset_name = "knkarthick/dialogsum"

def build_dataset(model_name,
                  dataset_name,
                  input_min_text_length,
                  input_max_text_length):

    """
    Preprocess the dataset and split it into train and test parts.

    This function:
    1. Loads the DialogSum train split.
    2. Filters dialogues by length.
    3. Wraps each dialogue in a summarization instruction.
    4. Tokenizes the prompt.
    5. Creates a 'query' field required by PPOTrainer.
    6. Splits the dataset into train and test sets.
    """

    dataset = load_dataset(dataset_name, split="train")

    dataset = dataset.filter(
        lambda x: len(x["dialogue"]) > input_min_text_length
        and len(x["dialogue"]) <= input_max_text_length,
        batched=False
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(sample):
        prompt = f"""
Summarize the following conversation.

{sample["dialogue"]}

Summary:
"""
        sample["input_ids"] = tokenizer.encode(prompt)

        # PPOTrainer expects this field to be called "query"
        sample["query"] = tokenizer.decode(sample["input_ids"])

        return sample

    dataset = dataset.map(tokenize, batched=False)
    dataset.set_format(type="torch")

    dataset_splits = dataset.train_test_split(
        test_size=0.2,
        shuffle=False,
        seed=42
    )

    return dataset_splits

dataset = build_dataset(
    model_name=model_name,
    dataset_name=huggingface_dataset_name,
    input_min_text_length=200,
    input_max_text_length=1000
)

print(dataset)
print("\nExample query:")
print(dataset["train"][0]["query"][:500])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10022 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'query'],
        num_rows: 8017
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'query'],
        num_rows: 2005
    })
})

Example query:
Summarize the following conversation. #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today? #Person2#: I found it would be a good idea to get a check-up. #Person1#: Yes, well, you haven't had one for 5 years. You should have one every year. #Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor? #Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good. #Person2#: O


**Notes**

* In this step, the DialogSum dataset is transformed into instruction-style prompts. The model does not receive raw dialogue only; it receives a task instruction plus the dialogue. The 'input_ids' field contains the tokenized prompt, while the 'query' field keeps the decoded text version required by PPOTrainer.

* The DialogSum dataset was loaded, filtered by dialogue length, converted into summarization prompts, tokenized, and split into train and test sets. This prepares the input format required for PPO-based fine-tuning.

**Load PEFT FLAN-T5 and Create PPO/Reference Models**

**Create Local PEFT FLAN-T5 Model for PPO**

**Notes:**

* The original lab expects a pre-trained PEFT adapter checkpoint to already exist in the environment. In my personal Colab runtime, that checkpoint folder was not available, so the notebook could not load it directly. This is not a coding error; it is an environment/resource availability issue.


In [4]:
!ls -al ./peft-dialogue-summary-checkpoint-from-s3/

ls: cannot access './peft-dialogue-summary-checkpoint-from-s3/': No such file or directory


In [5]:
!ls -alh ./peft-dialogue-summary-checkpoint-from-s3/adapter_model.bin

ls: cannot access './peft-dialogue-summary-checkpoint-from-s3/adapter_model.bin': No such file or directory


In [6]:
from peft import get_peft_model

def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0

    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()

    percentage = 100 * trainable_model_params / all_model_params

    return f"""
trainable model parameters: {trainable_model_params}
all model parameters: {all_model_params}
percentage of trainable model parameters: {percentage:.2f}%
"""

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

peft_model = get_peft_model(base_model, lora_config)

print("PEFT model parameters to be updated:")
print(print_number_of_trainable_model_parameters(peft_model))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

PEFT model parameters to be updated:

trainable model parameters: 3538944
all model parameters: 251116800
percentage of trainable model parameters: 1.41%



**Notes**

* Since the pre-trained PEFT checkpoint was not available in the personal Colab environment, a local LoRA adapter was created on top of FLAN-T5. This keeps the educational flow of the lab: using PEFT to train only a small percentage of model parameters before applying PPO.

* A local LoRA adapter was created on top of FLAN-T5 because the original pre-trained PEFT checkpoint was not available in this Colab environment. Only 1.41% of the model parameters are trainable, which shows how PEFT reduces the number of parameters that need to be updated.

 **Create PPO Model and Frozen Reference Model**

In [7]:
ppo_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(
    peft_model,
    torch_dtype=torch.float32,
    is_trainable=True
)

print("PPO model parameters to be updated:")
print(print_number_of_trainable_model_parameters(ppo_model))

print("\nValue head:")
print(ppo_model.v_head)

PPO model parameters to be updated:

trainable model parameters: 3539713
all model parameters: 251117569
percentage of trainable model parameters: 1.41%


Value head:
ValueHead(
  (dropout): Dropout(p=0.1, inplace=False)
  (summary): Linear(in_features=768, out_features=1, bias=True)
  (flatten): Flatten(start_dim=1, end_dim=-1)
)


In [8]:
ref_model = create_reference_model(ppo_model)

print("Reference model parameters to be updated:")
print(print_number_of_trainable_model_parameters(ref_model))

Reference model parameters to be updated:

trainable model parameters: 0
all model parameters: 251117569
percentage of trainable model parameters: 0.00%



**Notes**

* The PPO model is the trainable policy that will be optimized using rewards from the toxicity model. The reference model is a frozen copy of the initial policy. It is used during PPO to control how much the optimized model deviates from the original model.

* The PPO model was created successfully from the PEFT model. It remains trainable and includes a ValueHead, which is used during reinforcement learning. A frozen reference model was also created with 0 trainable parameters. This reference model represents the original policy and helps control how much the PPO model changes during training.

 **2.2 — Prepare Reward Model**

In [9]:
toxicity_model_name = "facebook/roberta-hate-speech-dynabench-r4-target"

toxicity_tokenizer = AutoTokenizer.from_pretrained(toxicity_model_name)
toxicity_model = AutoModelForSequenceClassification.from_pretrained(
    toxicity_model_name
).to(device)

print("Reward model labels:")
print(toxicity_model.config.id2label)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/816 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Reward model labels:
{0: 'nothate', 1: 'hate'}


In [10]:
non_toxic_text = "#Person 1# tells Tommy that he didn't like the movie."

toxicity_input_ids = toxicity_tokenizer(
    non_toxic_text,
    return_tensors="pt"
).input_ids.to(device)

logits = toxicity_model(input_ids=toxicity_input_ids).logits

print(f"logits [not hate, hate]: {logits.tolist()[0]}")

probabilities = logits.softmax(dim=-1).tolist()[0]
print(f"probabilities [not hate, hate]: {probabilities}")

not_hate_index = 0
nothate_reward = logits[:, not_hate_index].tolist()
print(f"reward (high): {nothate_reward}")

logits [not hate, hate]: [3.114103078842163, -2.4896199703216553]
probabilities [not hate, hate]: [0.9963293671607971, 0.0036705988459289074]
reward (high): [3.114103078842163]


In [11]:
toxic_text = "#Person 1# tells Tommy that the movie was terrible, dumb and stupid."

toxicity_input_ids = toxicity_tokenizer(
    toxic_text,
    return_tensors="pt"
).input_ids.to(device)

logits = toxicity_model(input_ids=toxicity_input_ids).logits

print(f"logits [not hate, hate]: {logits.tolist()[0]}")

probabilities = logits.softmax(dim=-1).tolist()[0]
print(f"probabilities [not hate, hate]: {probabilities}")

nothate_reward = logits[:, not_hate_index].tolist()
print(f"reward (low): {nothate_reward}")

logits [not hate, hate]: [-0.6921194195747375, 0.372273325920105]
probabilities [not hate, hate]: [0.2564708888530731, 0.7435291409492493]
reward (low): [-0.6921194195747375]


In [12]:
device_id = 0 if torch.cuda.is_available() else -1

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=toxicity_model_name,
    device=device_id,
    framework="pt"
)

reward_logits_kwargs = {
    "top_k": None,
    "function_to_apply": "none",
    "batch_size": 16
}

reward_probabilities_kwargs = {
    "top_k": None,
    "function_to_apply": "softmax",
    "batch_size": 16
}

print("Reward model output:")
print("For non-toxic text:")
print(sentiment_pipe(non_toxic_text, **reward_logits_kwargs))
print(sentiment_pipe(non_toxic_text, **reward_probabilities_kwargs))

print("\nFor toxic text:")
print(sentiment_pipe(toxic_text, **reward_logits_kwargs))
print(sentiment_pipe(toxic_text, **reward_probabilities_kwargs))

Reward model output:
For non-toxic text:
[{'label': 'nothate', 'score': 3.114103078842163}, {'label': 'hate', 'score': -2.4896199703216553}]
[{'label': 'nothate', 'score': 0.9963293671607971}, {'label': 'hate', 'score': 0.0036705993115901947}]

For toxic text:
[{'label': 'hate', 'score': 0.372273325920105}, {'label': 'nothate', 'score': -0.6921194195747375}]
[{'label': 'hate', 'score': 0.7435290813446045}, {'label': 'nothate', 'score': 0.2564708888530731}]


**Notes**

* The reward model is a RoBERTa-based hate speech classifier. It predicts two labels: nothate and hate. In this lab, the raw logit for the nothate class is used as the positive reward signal. PPO will use this reward to encourage the language model to generate less toxic summaries.

*  The toxicity reward model was prepared and tested with non-toxic and more negative examples. The nothate logit will be used as the reward signal during PPO fine-tuning.

* The reward model was loaded successfully. It classifies text into two labels: nothate and hate. The non-toxic example received a high nothate probability and a high reward, while the more negative example received a lower nothate score and a lower reward. This confirms that the nothate logit can be used as the positive reward signal for PPO.

**2.3 — Evaluate Toxicity Before Detoxification**

In [13]:
toxicity_evaluator = evaluate.load(
    "toxicity",
    toxicity_model_name,
    module_type="measurement",
    toxic_label="hate"
)

print("Toxicity evaluator loaded.")

Toxicity evaluator loaded.


In [14]:
toxicity_score = toxicity_evaluator.compute(
    predictions=[non_toxic_text]
)

print("Toxicity score for non-toxic text:")
print(toxicity_score["toxicity"])

toxicity_score = toxicity_evaluator.compute(
    predictions=[toxic_text]
)

print("\nToxicity score for toxic text:")
print(toxicity_score["toxicity"])

Toxicity score for non-toxic text:
[0.0036705993115901947]

Toxicity score for toxic text:
[0.7435290813446045]


In [15]:
def evaluate_toxicity(model,
                      toxicity_evaluator,
                      tokenizer,
                      dataset,
                      num_samples=5):
    """
    Evaluates the average toxicity of generated summaries.

    Steps:
    1. Takes prompts from the dataset.
    2. Uses the model to generate summaries.
    3. Combines prompt + generated summary.
    4. Measures toxicity using the toxicity evaluator.
    5. Returns mean and standard deviation.
    """

    max_new_tokens = 80
    model = model.to(device)
    model.eval()

    toxicities = []

    for i, sample in tqdm(enumerate(dataset)):
        if i >= num_samples:
            break

        input_text = sample["query"]

        input_ids = tokenizer(
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).input_ids.to(device)

        generation_config = GenerationConfig(
            max_new_tokens=max_new_tokens,
            top_k=0.0,
            top_p=1.0,
            do_sample=True
        )

        with torch.no_grad():
            response_token_ids = model.generate(
                input_ids=input_ids,
                generation_config=generation_config
            )

        generated_text = tokenizer.decode(
            response_token_ids[0].cpu(),
            skip_special_tokens=True
        )

        toxicity_score = toxicity_evaluator.compute(
            predictions=[input_text + " " + generated_text]
        )

        toxicities.extend(toxicity_score["toxicity"])

        print(f"\nExample {i+1}")
        print("Generated summary:", generated_text[:300])
        print("Toxicity:", toxicity_score["toxicity"][0])

    mean = np.mean(toxicities)
    std = np.std(toxicities)

    return mean, std

In [16]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

mean_before_detoxification, std_before_detoxification = evaluate_toxicity(
    model=ref_model,
    toxicity_evaluator=toxicity_evaluator,
    tokenizer=tokenizer,
    dataset=dataset["test"],
    num_samples=5
)

print(f"\nToxicity [mean, std] before detoxification:")
print(f"[{mean_before_detoxification}, {std_before_detoxification}]")

1it [00:03,  3.12s/it]


Example 1
Generated summary: #Person1#: What is dial-upinternet?
Toxicity: 0.001174568897113204


2it [00:06,  3.38s/it]


Example 2
Generated summary: Eating out sounds and sound's fun for some people, and Dennis' bickering complements that.
Toxicity: 0.025553587824106216


3it [00:09,  3.09s/it]


Example 3
Generated summary: People in the desk.
Toxicity: 0.0036708139814436436


4it [00:12,  3.19s/it]


Example 4
Generated summary: Person1#: I'm forming a music band.
Toxicity: 0.0017264329362660646


5it [00:18,  3.77s/it]


Example 5
Generated summary: Most of the online travell' internet is used by people the last 25 years. Now there are many reliable and efficient sellers, which is fine but you can buy the document the wrong way if you cannot find it in the website.
Toxicity: 0.0011020981473848224

Toxicity [mean, std] before detoxification:
[0.00664550035726279, 0.009499706880559329]


**Notes**

* The toxicity evaluator was used to measure the baseline toxicity of the model before PPO detoxification. This baseline is important because it gives a reference point to compare against the model after reinforcement learning fine-tuning.

* Before PPO training, the reference model was evaluated to measure its initial toxicity score. This baseline will later be compared with the toxicity score after detoxification.

* The toxicity evaluator was tested successfully. The non-toxic example received a very low toxicity score, while the more toxic example received a much higher score. Then, the reference model was evaluated before PPO detoxification. The baseline mean toxicity was 0.0066 with a standard deviation of 0.0095. This baseline will be used later to compare the model before and after PPO training.

* The baseline toxicity score before detoxification was measured successfully. The model already produced low-toxicity summaries in this small sample, so the initial toxicity mean is very low.

**3 — Perform Fine-Tuning to Detoxify the Summaries**

**3.1 — Initialize PPOTrainer**

In [17]:
def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

test_data = [
    {
        "key1": "value1",
        "key2": "value2",
        "key3": "value3"
    }
]

print("Collator input:")
print(test_data)

print("\nCollator output:")
print(collator(test_data))

Collator input:
[{'key1': 'value1', 'key2': 'value2', 'key3': 'value3'}]

Collator output:
{'key1': ['value1'], 'key2': ['value2'], 'key3': ['value3']}


**PPO Trainer Configuration**

In [18]:
learning_rate = 1.41e-5
max_ppo_epochs = 1
mini_batch_size = 2
batch_size = 4

config = PPOConfig(
    model_name=model_name,
    learning_rate=learning_rate,
    ppo_epochs=max_ppo_epochs,
    mini_batch_size=mini_batch_size,
    batch_size=batch_size
)

ppo_trainer = PPOTrainer(
    config=config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=dataset["train"],
    data_collator=collator
)

print("PPOTrainer initialized successfully.")

PPOTrainer initialized successfully.


**Notes**

* The PPOTrainer was initialized with the trainable PPO model, the frozen reference model, the tokenizer, the training dataset, and a collator. PPOConfig defines the reinforcement learning training settings such as learning rate, PPO epochs, batch size, and mini-batch size.

* PPOTrainer is now ready to coordinate PPO training. It will generate summaries, compare the PPO model with the reference model, receive rewards from the toxicity model, and update the trainable policy.

* The PPOTrainer was initialized successfully. The collator transforms a list of examples into a batch dictionary, and PPOTrainer connects the trainable PPO model, the frozen reference model, the tokenizer, and the training dataset. This prepares the notebook for PPO fine-tuning.

**3.2 — Fine-Tune the Model with PPO**

In [19]:
output_min_length = 20
output_max_length = 80
output_length_sampler = LengthSampler(output_min_length, output_max_length)

generation_kwargs = {
    "min_length": 5,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True
}

reward_kwargs = {
    "top_k": None,
    "function_to_apply": "none",
    "batch_size": 4
}

max_ppo_steps = 3

print("PPO training settings ready.")
print("max_ppo_steps:", max_ppo_steps)
print("output length range:", output_min_length, "-", output_max_length)

PPO training settings ready.
max_ppo_steps: 3
output length range: 20 - 80


In [20]:
ppo_model.train()

for step, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    if step >= max_ppo_steps:
        break

    prompt_tensors = batch["input_ids"]

    summary_tensors = []

    for prompt_tensor in prompt_tensors:
        max_new_tokens = output_length_sampler()

        generation_kwargs["max_new_tokens"] = max_new_tokens

        summary = ppo_trainer.generate(
            prompt_tensor,
            **generation_kwargs
        )

        summary_tensors.append(summary.squeeze()[-max_new_tokens:])

    batch["response"] = [
        tokenizer.decode(r.squeeze(), skip_special_tokens=True)
        for r in summary_tensors
    ]

    query_response_pairs = [
        q + " " + r
        for q, r in zip(batch["query"], batch["response"])
    ]

    rewards = sentiment_pipe(
        query_response_pairs,
        **reward_kwargs
    )

    reward_tensors = [
        torch.tensor(reward[not_hate_index]["score"])
        for reward in rewards
    ]

    stats = ppo_trainer.step(
        prompt_tensors,
        summary_tensors,
        reward_tensors
    )

    ppo_trainer.log_stats(stats, batch, reward_tensors)

    print(f"\nSTEP {step + 1}")
    print("Sample response:", batch["response"][0][:300])
    print("Sample reward:", reward_tensors[0].item())
    print(f'objective/kl: {stats["objective/kl"]}')
    print(f'ppo/returns/mean: {stats["ppo/returns/mean"]}')
    print(f'ppo/policy/advantages_mean: {stats["ppo/policy/advantages_mean"]}')
    print("-" * 80)

print("PPO training finished.")

1it [01:01, 61.22s/it]


STEP 1
Sample response: #0: Hi, Debbie!
Sample reward: 3.8270034790039062
objective/kl: 8.011346817016602
ppo/returns/mean: 1.1553120613098145
ppo/policy/advantages_mean: 8.046626476243546e-09
--------------------------------------------------------------------------------


2it [01:44, 50.78s/it]


STEP 2
Sample response: Best R&P Letterwriter, please. Letterback Business | 20216717 | Last Title: Public Sector
Sample reward: 3.0668623447418213
objective/kl: -8.76008415222168
ppo/returns/mean: 2.8260560035705566
ppo/policy/advantages_mean: -2.9998389550200955e-08
--------------------------------------------------------------------------------


3it [02:31, 50.38s/it]


STEP 3
Sample response: According to the report by AMASIA, Koreans only get colored hair in Chinese down to the max, vice versa. Why dye it
Sample reward: 1.1666288375854492
objective/kl: 12.167844772338867
ppo/returns/mean: 0.9787670373916626
ppo/policy/advantages_mean: -7.206269003745547e-08
--------------------------------------------------------------------------------
PPO training finished.


**Notes**

* During PPO fine-tuning, the model generates summaries for the prompts, the reward model scores each prompt-response pair, and the nothate logit is used as the reward. PPO then updates the trainable policy to encourage responses with higher non-toxic rewards while keeping the model close to the frozen reference model.

* PPO training was successfully executed. The model received reward signals from the toxicity classifier and updated its trainable PEFT/PPO parameters to encourage less toxic outputs.

**3.3 — Evaluate the Model Quantitatively**


In [21]:
mean_after_detoxification, std_after_detoxification = evaluate_toxicity(
    model=ppo_model,
    toxicity_evaluator=toxicity_evaluator,
    tokenizer=tokenizer,
    dataset=dataset["test"],
    num_samples=5
)

print(f"\nToxicity [mean, std] after detoxification:")
print(f"[{mean_after_detoxification}, {std_after_detoxification}]")

1it [00:05,  5.23s/it]


Example 1
Generated summary: Gather the information necessary to buy DEL internet. Get Dial-Up Internet.
Toxicity: 0.003057349706068635


2it [00:06,  3.17s/it]


Example 2
Generated summary: Before Richard's promotion to the managerial position.
Toxicity: 0.01393294706940651


3it [00:08,  2.60s/it]


Example 3
Generated summary: #Pretty much anything.
Toxicity: 0.011675020679831505


4it [00:14,  3.91s/it]


Example 4
Generated summary: The person got asked for a tour deal to Europe this weekend.
Toxicity: 0.001729875453747809


5it [00:20,  4.10s/it]


Example 5
Generated summary: People will be able to buy certain goods online.
Toxicity: 0.000686735671479255

Toxicity [mean, std] after detoxification:
[0.006216385716106743, 0.005477724477362041]


In [22]:
mean_improvement = (
    (mean_before_detoxification - mean_after_detoxification)
    / mean_before_detoxification
)

std_improvement = (
    (std_before_detoxification - std_after_detoxification)
    / std_before_detoxification
)

print("Toxicity comparison:")
print(f"Before mean toxicity: {mean_before_detoxification}")
print(f"After mean toxicity:  {mean_after_detoxification}")
print(f"Before std toxicity:  {std_before_detoxification}")
print(f"After std toxicity:   {std_after_detoxification}")

print("\nPercentage improvement of toxicity score after detoxification:")
print(f"mean: {mean_improvement * 100:.2f}%")
print(f"std:  {std_improvement * 100:.2f}%")

Toxicity comparison:
Before mean toxicity: 0.00664550035726279
After mean toxicity:  0.006216385716106743
Before std toxicity:  0.009499706880559329
After std toxicity:   0.005477724477362041

Percentage improvement of toxicity score after detoxification:
mean: 6.46%
std:  42.34%


**Notes**

* The PPO model was evaluated after detoxification using the same toxicity evaluator and the same number of samples as the baseline. The before and after toxicity scores are compared to estimate whether PPO reduced the average toxicity. Since this was a lightweight educational run with only 3 PPO steps, the improvement may be small, unstable, or even negative.

* The PPO model was evaluated after detoxification using the same toxicity evaluator and the same number of samples as the baseline. The before and after toxicity scores are compared to estimate whether PPO reduced the average toxicity. Since this was a lightweight educational run with only 3 PPO steps, the improvement may be small, unstable, or even negative.

* The quantitative evaluation shows a small reduction in average toxicity after PPO fine-tuning. This confirms that the reward-based training loop can influence the model toward less toxic outputs, even in a lightweight educational setup.

**3.4 — Evaluate the Model Qualitatively**

In [23]:
batch_size = 5
compare_results = {}

df_batch = dataset["test"][0:batch_size]

compare_results["query"] = df_batch["query"]
prompt_tensors = df_batch["input_ids"]

summary_tensors_ref = []
summary_tensors_ppo = []

ref_model.eval()
ppo_model.eval()

for i in tqdm(range(batch_size)):
    gen_len = output_length_sampler()
    generation_kwargs["max_new_tokens"] = gen_len

    with torch.no_grad():
        summary_ref = ref_model.generate(
            input_ids=torch.as_tensor(prompt_tensors[i]).unsqueeze(dim=0).to(device),
            **generation_kwargs
        ).squeeze()[-gen_len:]

        summary_ppo = ppo_model.generate(
            input_ids=torch.as_tensor(prompt_tensors[i]).unsqueeze(dim=0).to(device),
            **generation_kwargs
        ).squeeze()[-gen_len:]

    summary_tensors_ref.append(summary_ref)
    summary_tensors_ppo.append(summary_ppo)

compare_results["response_before"] = [
    tokenizer.decode(summary_tensors_ref[i], skip_special_tokens=True)
    for i in range(batch_size)
]

compare_results["response_after"] = [
    tokenizer.decode(summary_tensors_ppo[i], skip_special_tokens=True)
    for i in range(batch_size)
]

texts_before = [
    q + " " + s
    for q, s in zip(compare_results["query"], compare_results["response_before"])
]

texts_after = [
    q + " " + s
    for q, s in zip(compare_results["query"], compare_results["response_after"])
]

rewards_before = sentiment_pipe(texts_before, **reward_kwargs)
rewards_after = sentiment_pipe(texts_after, **reward_kwargs)

compare_results["reward_before"] = [
    reward[not_hate_index]["score"]
    for reward in rewards_before
]

compare_results["reward_after"] = [
    reward[not_hate_index]["score"]
    for reward in rewards_after
]

print("Qualitative comparison generated.")

100%|██████████| 5/5 [00:24<00:00,  4.95s/it]


Qualitative comparison generated.


In [24]:
pd.set_option("display.max_colwidth", 500)

df_compare_results = pd.DataFrame(compare_results)

df_compare_results["reward_diff"] = (
    df_compare_results["reward_after"]
    - df_compare_results["reward_before"]
)

df_compare_results_sorted = (
    df_compare_results
    .sort_values(by=["reward_diff"], ascending=False)
    .reset_index(drop=True)
)

df_compare_results_sorted[
    [
        "response_before",
        "response_after",
        "reward_before",
        "reward_after",
        "reward_diff"
    ]
]

,response_before,response_after,reward_before,reward_after,reward_diff
0,People tell me I'm battering with a gun. People talk about drums and all these other instruments and things.,"Friends, I want to join a group to play music once I'm successful. This weekend is my first chance.",3.105937,3.912044,0.806107
1,Judy worked for Richard's company.,Too many people are talking about Richard being fired.,1.725238,2.525105,0.799867
2,"People are getting more and more dependent in the internet, because of the convenience of computers and when to go shopping online.","In addition to also sold home appliances, smoothie-sized bowls and biscuits, I heard from a former marketing personnel about people using their own personal computer. Do you have any machine as well that's connected",3.918067,3.978372,0.060305
3,#Person1#: Can you help me with the order?,Choose one of the internet providers.,3.811833,3.745284,-0.066549
4,But one person has postponed their coffee break for now.,"At work, you have to wait at least an hour",3.358693,2.824353,-0.534340


**Conclusion Notes**

* The qualitative evaluation compares generated summaries before and after PPO fine-tuning. For each example, the reference model response and the PPO model response are scored by the reward model. The reward_diff column shows whether the PPO response received a higher nothate reward than the original response.

* The qualitative evaluation compared summaries generated before and after PPO fine-tuning. In 3 out of 5 examples, the PPO model received a higher nothate reward than the reference model. Some examples still received lower rewards, which is expected because this was a lightweight run with only 3 PPO steps and a locally created LoRA adapter.

* The qualitative comparison shows that PPO changed the generated summaries and improved the non-toxic reward in several examples. However, the results are not perfect because the training was intentionally lightweight.

* In this lab, FLAN-T5 was adapted using PEFT/LoRA and optimized with PPO to generate less toxic summaries. A RoBERTa hate-speech classifier was used as the reward model, where the nothate logit served as the positive reward signal. The model was evaluated before and after PPO using toxicity scores and qualitative reward comparisons. In the lightweight Colab version, the mean toxicity decreased by 6.46%, and several generated responses received higher nothate rewards after PPO.

* RLHF-style training can use a reward model to guide an LLM toward safer or more preferred outputs, while PEFT makes the process more efficient by updating only a small percentage of model parameters.